In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import torch
import torch.nn as nn
import math

class Transpose(nn.Module):
    def __init__(self, dim0, dim1):
        super().__init__()
        self.dim0 = dim0
        self.dim1 = dim1

    def forward(self, x):
        return x.transpose(self.dim0, self.dim1)

class PositionalEncoding2D(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.register_buffer('pos_embed', self.build_sincos_encoding(num_patches, dim), persistent=False)

    def build_sincos_encoding(self, num_patches, dim):
        pe = torch.zeros(num_patches, dim)
        position = torch.arange(0, num_patches, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # [1, num_patches, dim]

    def forward(self, x):
        return x + self.pos_embed[:, :x.size(1), :]

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels)
        )
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return self.block(x) + self.skip(x)

class CondEncoder(nn.Module):
    def __init__(self, in_channels=4, out_channels=736, num_tokens=64):
        super().__init__()
        self.encoder = nn.Sequential(
            ResidualBlock(in_channels, 64), # [B, 64, 64, 64]
            nn.AvgPool2d(2), # [B, 64, 32, 32]
            ResidualBlock(64, 128),
            nn.AvgPool2d(2), # [B, 128, 16, 16]
            ResidualBlock(128, 256),
            nn.AvgPool2d(2), # [B, 256, 8, 8]
            nn.Conv2d(256, out_channels, kernel_size=1) # [B, 736, 8, 8]
        )
        self.proj = nn.Sequential(
            nn.Flatten(2),  # [B, 736, 64]
            Transpose(-1, -2),   # [B, 64, 736]
        )
        self.pos_embed = PositionalEncoding2D(num_patches=num_tokens, dim=out_channels)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        feat = self.encoder(x)          # [B, 736, 8, 8]
        tokens = self.proj(feat)        # [B, 64, 736]
        tokens = self.pos_embed(tokens) # [B, 64, 736]
        tokens = self.norm(tokens)
        return tokens


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from diffusers import DDPMScheduler, UNet2DConditionModel
from accelerate import Accelerator
import os
from PIL import Image
from torchvision import transforms
import numpy as np
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import StepLR
from diffusers import AutoencoderKL

class OutpaintTrainer:
    def __init__(self, pretrained_path="runwayml/stable-diffusion-v1-5"):
        # Initialize Accelerator for FP16 mixed precision
        self.accelerator = Accelerator(mixed_precision='fp16')
        self.loss_history = []
        self.val_loss_history = []

        # Load model components
        self.vae = AutoencoderKL.from_pretrained(pretrained_path, subfolder="vae")
        self.unet = UNet2DConditionModel.from_pretrained(pretrained_path, subfolder="unet")

        # Coordinate Encoder
        self.coord_encoder = nn.Sequential(nn.Linear(4, 32), nn.GELU())

        # Conditional Projection Layer

        self.cond_proj = CondEncoder()
        # Prepare components for mixed precision
        components = [self.vae, self.unet, self.coord_encoder, self.cond_proj]
        self.vae, self.unet, self.coord_encoder, self.cond_proj = \
            self.accelerator.prepare(*components)

        # Optimizer
        self.optimizer = torch.optim.AdamW(
            list(self.unet.parameters()) +
            list(self.coord_encoder.parameters()) +
            list(self.cond_proj.parameters()),
            lr = 2e-5)
        self.optimizer = self.accelerator.prepare(self.optimizer)

        # Noise Scheduler
        self.noise_scheduler = DDPMScheduler.from_pretrained(
            pretrained_path, subfolder="scheduler")

        # Freeze VAE
        self.vae.requires_grad_(False)
        self.accelerator.register_for_checkpointing(self.unet, self.coord_encoder, self.cond_proj, self.optimizer)


In [4]:
import torch.nn as nn
import torch
import matplotlib.pyplot as plt
from diffusers import DDPMScheduler, UNet2DConditionModel
from accelerate import Accelerator
from torchvision import transforms
from PIL import Image
from diffusers import AutoencoderKL

cnt = 0

class OutpaintEngine:
    def __init__(self, device="cuda"):
        """
        Initialize the outpainting engine.

        Args:
            device: Computation device (default: "cuda").
        """
        self.device = device

        # Load model components
        self.vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-ema").to(self.device)
        self.unet = UNet2DConditionModel.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            subfolder="unet"
        )

        self.accelerator = Accelerator(
            mixed_precision='fp16'
        )

        # Initialize scheduler
        self.noise_scheduler = DDPMScheduler.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            subfolder="scheduler"
        )

        # Direction encoder
        self.directions = ['right', 'left', 'down', 'up']
        self.coord_encoder = nn.Sequential(nn.Linear(4, 32), nn.GELU())

        # Condition projection layer

        self.cond_proj = CondEncoder()

        # Set eval mode
        self.vae.eval()
        self.vae.to(device)
        self.unet.eval()
        self.unet.to(device)
        self.cond_proj.eval()
        self.cond_proj.to(device)
        self.coord_encoder.eval()
        self.coord_encoder.to(device)

        self.transform = transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3)
        ])

    def _extract_old_region(self, image_tensor, direction, crop_ratio):
        """Apply zero mask to input image
        Args:
            image_tensor: Input tensor (1,3,H,W)
            direction: Mask direction (0-3)
            crop_ratio: Ratio of preserved area
        Returns:
            masked_tensor: Zero-masked image tensor
            mask_params: Parameters for latent mask calculation
        """
        b, c, h, w = image_tensor.shape
        masked = image_tensor.clone()

        if direction in [0, 1]:  # Horizontal directions
            crop_w = int(w * crop_ratio)
            if direction == 0:  # Preserve left
                masked = masked[..., :, :, :crop_w]
            else:  # Preserve right
                masked = masked[..., :, :, w - crop_w:]
        else:  # Vertical directions
            crop_h = int(h * crop_ratio)
            if direction == 2:  # Preserve top
                masked = masked[..., :crop_h, :]
            else:  # Preserve bottom
                masked = masked[..., h - crop_h:, :]

        return masked

    def _create_latent_mask_from_bbox(self, bbox, latent_shape):
        b, _, lh, lw = latent_shape
        masks = []
        for coords in bbox:
            x1 = int(coords[0] * lw)
            y1 = int(coords[1] * lh)
            x2 = int(coords[2] * lw)
            y2 = int(coords[3] * lh)
            mask = torch.zeros((1, lh, lw), device=self.device)
            mask[:, x1:x2, y1:y2] = 1
            masks.append(mask)
        return torch.stack(masks)

    def _create_latent_mask(self, bbox, latent_shape):
        b, _, lh, lw = latent_shape
        masks = []
        for coords in bbox:
            x1 = coords[0] * lw
            y1 = coords[1] * lh
            x2 = coords[2] * lw
            y2 = coords[3] * lh

            xx, yy = torch.meshgrid(
                torch.arange(lw, device=self.accelerator.device),
                torch.arange(lh, device=self.accelerator.device))
            mask = ((xx >= x1) & (xx <= x2) & (yy >= y1) & (yy <= y2)).float()

            masks.append(mask)
        return torch.stack(masks).unsqueeze(1)

    def compute_mean_std(self, x):
        mean = x.mean(dim=(2, 3), keepdim=True)  # [1, 4, 1, 1]
        std = x.std(dim=(2, 3), keepdim=True)  # [1, 4, 1, 1]
        return mean, std

    def generate_iterative(self, image_tensor, steps=200, crop_ratio=0.97, iterations=10, direction="right", name="default"):
        """
        Iteratively expand image along a direction using bbox + coord_encoder logic.
        """
        direction_idx = self.directions.index(direction)
        current_image = image_tensor.clone()
        b, c, h, w = current_image.shape
        lh, lw = int(64), int(64)
        #initial_latent = self.vae.encode(current_image).latent_dist.sample()
        #initial_latent = initial_latent * self.vae.config.scaling_factor
        initial_mean, initial_std = self.compute_mean_std(current_image)

        crop_w, crop_h = int(w * crop_ratio), int(h * crop_ratio)
        crop_lw, crop_lh = int(lw * crop_ratio), int(lh * crop_ratio)
        mask_size = w - crop_w if direction in ["left", "right"] else h - crop_h
        latent_mask_size = lw - crop_lw if direction in ["left", "right"] else lh - crop_lh

        # Extract preserved region to initialize stitched image
        preserved_region = self._extract_old_region(current_image, direction_idx, crop_ratio)
        stitched = preserved_region
        current_latent = None
        # Step 1: Create bbox (normalized)
        if direction == "right":
            bbox = torch.tensor([[0.0, crop_ratio, 1.0, 1.0]], device=self.device)
        elif direction == "left":
            bbox = torch.tensor([[0.0, 0.0, 1.0, 1.0 - crop_ratio]], device=self.device)
        elif direction == "down":
            bbox = torch.tensor([[crop_ratio, 0.0, 1.0, 1.0]], device=self.device)
        else:
            bbox = torch.tensor([[0.0, 0.0, 1.0 - crop_ratio, 1.0]], device=self.device)

        for i in range(iterations):
            print(f"[{i + 1}/{iterations}] Expanding → {direction.upper()}")
            if i == 0:
                # Step 2: Create masked image
                mask = torch.zeros_like(current_image)
                x1 = int(bbox[0][0] * w)
                y1 = int(bbox[0][1] * h)
                x2 = int(bbox[0][2] * w)
                y2 = int(bbox[0][3] * h)
                mask[:, :, x1:x2, y1:y2] = 1
                masked_img = current_image * (1 - mask)
                #print(mask)

                # Step 3: Encode condition
                with torch.no_grad():
                    masked_latents = self.vae.encode(masked_img).latent_dist.sample()
                    masked_latents = masked_latents * self.vae.config.scaling_factor
            else:
                masked_latents = current_latent

            # Step 4: Add noise and create latent_mask
            latent_mask = self._create_latent_mask(bbox, masked_latents.shape)
            #print(latent_mask)

            noise = torch.randn_like(masked_latents)
            noisy_latents = self.noise_scheduler.add_noise(masked_latents * latent_mask, noise * latent_mask,
                                                           torch.tensor(steps))
            noisy_latents = masked_latents * (1 - latent_mask) + noisy_latents * latent_mask

            # Step 5: Denoising loop
            self.noise_scheduler.set_timesteps(steps)
            latent_input = noisy_latents

            condition = torch.cat([
                self.cond_proj(masked_latents),
                self.coord_encoder(bbox).unsqueeze(1).expand(-1, 64, -1)
            ], dim=-1)

            for t in self.noise_scheduler.timesteps:
                latent_input = latent_input * latent_mask + masked_latents * (1 - latent_mask)
                with torch.no_grad():
                    noise_pred = self.unet(latent_input, t, encoder_hidden_states=condition).sample
                latent_input = self.noise_scheduler.step(noise_pred, t, latent_input).prev_sample

            # Step 6: Decode image
            with torch.no_grad():
                generated_latent = masked_latents * (1 - latent_mask) + latent_input * latent_mask
                test = masked_latents * (1 - latent_mask)
                generated_img = self.vae.decode(generated_latent / self.vae.config.scaling_factor).sample
                test = self.vae.decode(test / self.vae.config.scaling_factor).sample

            #plt.figure(figsize=(8, 8))
            #plt.imshow((test[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5).clip(0, 1))
            #plt.title(f"Test {i + 1} Result")
            #plt.axis("off")
            #plt.show()

            #tgt_mean, tgt_std = self.compute_mean_std(generated_img)
            #generated_img = (generated_img - tgt_mean) / tgt_std * initial_std + initial_mean
            # Step 7: Extract new patch and update
            plt.figure(figsize=(8, 8))
            plt.imshow((generated_img[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5).clip(0, 1))
            torch.save((generated_img[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5).clip(0, 1),
                       f'predicted_cell_map{i + 1}.pt')
            #plt.title(f"Generated {i + 1} Result")
            plt.axis("off")
            if i == 0:
                plt.savefig(f'i.png')
            plt.show()

            new_patch = self._extract_new_region(generated_img, direction_idx, mask_size)
            #if is_patch_mostly_white(new_patch, threshold=0.9):
            #  i-=1
            #  print("White patch detected, regenerating...")
            #  continue
            stitched = self._stitch_image(stitched, new_patch, direction_idx)

            # save current_latent 和 current_image
            latent_save_path = f"drive/MyDrive/outinfer/{name}/{i:02d}_latent.pt"
            image_save_path = f"drive/MyDrive/outinfer/{name}/{i:02d}_image.png"
            os.makedirs(os.path.dirname(latent_save_path), exist_ok=True)
            torch.save(generated_latent.cpu(), latent_save_path)  # 保存为 half 精度节省空间
            current_img = (generated_img[0].clamp(-1, 1) * 0.5 + 0.5).cpu()
            Image.fromarray((current_img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)).save(image_save_path)

            current_image = self._cyclic_shift(generated_img, direction_idx, mask_size)
            current_latent = self._cyclic_latent_shift(generated_latent, direction_idx, latent_mask_size)

            # Display intermediate
            plt.figure(figsize=(8, 8))
            plt.imshow((stitched[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5).clip(0, 1))
            #plt.title(f"Iteration {i + 1} Result")
            plt.axis("off")
            plt.show()



        return ((stitched[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5).clip(0, 1) * 255).astype(np.uint8)

    def _extract_new_region(self, generated, direction, mask_size):
        """extract new region"""
        if direction == 0:  # right
            return generated[..., :, :, -mask_size:]
        elif direction == 1:  # left
            return generated[..., :, :, :mask_size]
        elif direction == 2:  # down
            return generated[..., :, -mask_size:, :]
        elif direction == 3:  # up
            return generated[..., :, :mask_size, :]

    def _extract_new_latent_region(self, generated, direction, mask_size):
        """extract new region"""

        if direction == 0:  # right
            return generated[..., :, :, -mask_size:]
        elif direction == 1:  # left
            return generated[..., :, :, :mask_size]
        elif direction == 2:  # down
            return generated[..., :, -mask_size:, :]
        elif direction == 3:  # up
            return generated[..., :, :mask_size, :]

    def _stitch_image(self, combined, generated_patch, direction):
        """
        Append newly generated regions to the stitched image to create a final large output.

        Args:
            combined: The accumulated image tensor containing all past expansions.
            generated_patch: The new generated patch to append.
            direction: The expansion direction.

        Returns:
            Updated stitched image tensor.
        """
        if direction == 0:
            combined = torch.cat([combined, generated_patch], dim=-1)
        elif direction == 1:
            combined = torch.cat([generated_patch, combined], dim=-1)
        elif direction == 2:
            combined = torch.cat([combined, generated_patch], dim=-2)
        elif direction == 3:
            combined = torch.cat([generated_patch, combined], dim=-2)

        return combined

    def _cyclic_shift(self, generated, direction, mask_size):
        """do iteration（"for example generating right region"：ABCDE → BCDEA）"""
        if direction == 0:  # right
            return torch.cat([generated[..., :, :, mask_size:],
                              generated[..., :, :, :mask_size]], dim=-1)
        elif direction == 1:  # left
            return torch.cat([generated[..., :, :, -mask_size:],
                              generated[..., :, :, :-mask_size]], dim=-1)
        elif direction == 2:  # down
            return torch.cat([generated[..., :, mask_size:, :],
                              generated[..., :, :mask_size, :]], dim=-2)
        elif direction == 3:  # up
            return torch.cat([generated[..., :, -mask_size:, :],
                              generated[..., :, :-mask_size, :]], dim=-2)

    def _cyclic_latent_shift(self, generated, direction, mask_size):
        """do iteration（"for example generating right region"：ABCDE → BCDEA）"""
        if direction == 0:  # right
            return torch.cat([generated[..., :, :, mask_size:],
                              generated[..., :, :, :mask_size]], dim=-1)
        elif direction == 1:  # left
            return torch.cat([generated[..., :, :, -mask_size:],
                              generated[..., :, :, :-mask_size]], dim=-1)
        elif direction == 2:  # down
            return torch.cat([generated[..., :, mask_size:, :],
                              generated[..., :, :mask_size, :]], dim=-2)
        elif direction == 3:  # up
            return torch.cat([generated[..., :, -mask_size:, :],
                              generated[..., :, :-mask_size, :]], dim=-2)


In [ ]:
trainer = OutpaintTrainer()
trainer.accelerator.load_state("drive/MyDrive/checkpoint_LR")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

In [ ]:
outpaint_engine = OutpaintEngine()
outpaint_engine.unet = trainer.unet
outpaint_engine.cond_proj = trainer.cond_proj
outpaint_engine.coord_encoder = trainer.coord_encoder

In [ ]:
import torch
import matplotlib.pyplot as plt
import os
import numpy as np

def plot_all_param_histograms(model, bins=100, save_dir=None, exclude_bias=True):
    """
    绘制模型所有参数的直方图分布图，并标注均值和标准差。

    Args:
        model: PyTorch 模型
        bins: 直方图分箱数量
        save_dir: 如果不为 None，则保存图像而不是显示
        exclude_bias: 是否跳过 bias 参数
    """
    for name, param in model.named_parameters():
        if exclude_bias and "bias" in name:
            continue

        data = param.detach().cpu().flatten().numpy()
        mean = np.mean(data)
        std = np.std(data)

        plt.figure(figsize=(6, 4))
        plt.hist(data, bins=bins, color='skyblue', edgecolor='black')
        plt.title(f"Histogram of {name}", fontsize=12)
        plt.xlabel("Value")
        plt.ylabel("Frequency")

        # 添加均值和标准差的文字
        plt.text(0.98, 0.95,
                 f"mean = {mean:.4e}\nstd = {std:.4e}",
                 horizontalalignment='right',
                 verticalalignment='top',
                 transform=plt.gca().transAxes,
                 fontsize=10,
                 bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))

        plt.tight_layout()

        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            save_path = os.path.join(save_dir, f"{name.replace('.', '_')}.png")
            plt.savefig(save_path, dpi=150)
            plt.close()
        else:
            plt.show()


In [ ]:
trainer = OutpaintTrainer()
trainer.accelerator.load_state("drive/MyDrive/checkpoint_25")
bad = trainer.unet
trainer1 = OutpaintTrainer()
trainer1.accelerator.load_state("drive/MyDrive/checkpoint_LR")
good = trainer1.unet

In [ ]:
plot_all_param_histograms(trainer.cond_proj)

In [ ]:
plot_all_param_histograms(trainer1.cond_proj)

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def collect_param_stats(model, label):
    stats = []
    for name, param in model.named_parameters():
        if param.ndim > 0:
            data = param.detach().cpu().flatten().numpy()
            stats.append({"Layer": name, "Stat": "Std", "Value": data.std(), "Model": label})
    return stats

df = pd.DataFrame(collect_param_stats(trainer1.cond_proj, "Good") + collect_param_stats(trainer.cond_proj, "Bad"))


plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="Stat", y="Value", hue="Model", palette=["green", "red"])
plt.title("Parameter Mean and Std Distribution (Boxplot)")
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
from PIL import Image
if __name__ == "__main__":
    # Preprocess input image
    root_dir = "drive/MyDrive/outtest"
    img_files = [os.path.join(root_dir, f) for f in os.listdir(root_dir)]
    for i in img_files:
      for j in range(5):
        img = Image.open(i).convert("RGB")
        img = outpaint_engine.transform(img).unsqueeze(0).to(outpaint_engine.accelerator.device)
        stitched_image = outpaint_engine.generate_iterative(img, steps=200,crop_ratio=0.95, iterations=15, direction="up", name=str(i)+str(j))
        os.makedirs(os.path.dirname(f"drive/MyDrive/outinfer/{i}.png"), exist_ok=True)
        Image.fromarray(stitched_image).save(f"drive/MyDrive/outinfer/{i}+{j}.png")
